In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_long = pd.read_csv('C:\\Users\\00000\\oasis_longitudinal.csv')
data_long.head()

,Subject ID,MRI ID,Group,Visit,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,OAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,OAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,OAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,OAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,OAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [3]:
nan_df = data_long[data_long['SES'].isna()]
print(nan_df)

    Subject ID         MRI ID     Group  Visit  MR Delay M/F Hand  Age  EDUC  \
2    OAS2_0002  OAS2_0002_MR1  Demented      1         0   M    R   75    12   
3    OAS2_0002  OAS2_0002_MR2  Demented      2       560   M    R   76    12   
4    OAS2_0002  OAS2_0002_MR3  Demented      3      1895   M    R   80    12   
10   OAS2_0007  OAS2_0007_MR1  Demented      1         0   M    R   71    16   
11   OAS2_0007  OAS2_0007_MR3  Demented      3       518   M    R   73    16   
12   OAS2_0007  OAS2_0007_MR4  Demented      4      1281   M    R   75    16   
134  OAS2_0063  OAS2_0063_MR1  Demented      1         0   F    R   80    12   
135  OAS2_0063  OAS2_0063_MR2  Demented      2       490   F    R   81    12   
207  OAS2_0099  OAS2_0099_MR1  Demented      1         0   F    R   80    12   
208  OAS2_0099  OAS2_0099_MR2  Demented      2       807   F    R   83    12   
237  OAS2_0114  OAS2_0114_MR1  Demented      1         0   F    R   76    12   
238  OAS2_0114  OAS2_0114_MR2  Demented 

In [4]:
summary = data_long.select_dtypes(include='number').agg(['min', 'max', 'median'])
print(summary)

        Visit  MR Delay   Age  EDUC  SES  MMSE  CDR    eTIV   nWBV    ASF
min       1.0       0.0  60.0   6.0  1.0   4.0  0.0  1106.0  0.644  0.876
max       5.0    2639.0  98.0  23.0  5.0  30.0  2.0  2004.0  0.837  1.587
median    2.0     552.0  77.0  15.0  2.0  29.0  0.0  1470.0  0.729  1.194


In [5]:
CDR_mapping = { 
    0: 0, 
    0.5: 1, 
    1: 2, 
    2: 3
}

data_long['CDR'] = data_long['CDR'].replace(CDR_mapping)

In [6]:
data_selected = data_long[["M/F", "Age", "EDUC", "SES", "MMSE", "eTIV", "nWBV", "CDR"]]

In [7]:
X = data_selected.drop('CDR', axis=1)
y = data_selected['CDR']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [9]:
median_ses = X_train['SES'].median()
median_mmse = X_train['MMSE'].median()

In [10]:
X_train['SES'] = X_train['SES'].fillna(median_ses)
X_test['SES'] = X_test['SES'].fillna(median_ses)
X_train['MMSE'] = X_train['MMSE'].fillna(median_mmse)
X_test['MMSE'] = X_test['MMSE'].fillna(median_mmse)
print(data_long[data_long['SES'].isna()])
print(data_long[data_long['MMSE'].isna()])

    Subject ID         MRI ID     Group  Visit  MR Delay M/F Hand  Age  EDUC  \
2    OAS2_0002  OAS2_0002_MR1  Demented      1         0   M    R   75    12   
3    OAS2_0002  OAS2_0002_MR2  Demented      2       560   M    R   76    12   
4    OAS2_0002  OAS2_0002_MR3  Demented      3      1895   M    R   80    12   
10   OAS2_0007  OAS2_0007_MR1  Demented      1         0   M    R   71    16   
11   OAS2_0007  OAS2_0007_MR3  Demented      3       518   M    R   73    16   
12   OAS2_0007  OAS2_0007_MR4  Demented      4      1281   M    R   75    16   
134  OAS2_0063  OAS2_0063_MR1  Demented      1         0   F    R   80    12   
135  OAS2_0063  OAS2_0063_MR2  Demented      2       490   F    R   81    12   
207  OAS2_0099  OAS2_0099_MR1  Demented      1         0   F    R   80    12   
208  OAS2_0099  OAS2_0099_MR2  Demented      2       807   F    R   83    12   
237  OAS2_0114  OAS2_0114_MR1  Demented      1         0   F    R   76    12   
238  OAS2_0114  OAS2_0114_MR2  Demented 

In [11]:
label_encoder_MF = LabelEncoder()
X_train['M/F'] = label_encoder_MF.fit_transform(X_train['M/F'])
X_test['M/F'] = label_encoder_MF.transform(X_test['M/F'])

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth = 6, criterion = 'entropy', max_leaf_nodes = 4)
cv_scores = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='accuracy')
print("Decision tree accuracy:", np.mean(cv_scores))

Decision tree accuracy: 0.7419209039548023


In [14]:
rf_model = RandomForestClassifier(random_state=42, criterion = 'gini')
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
print("Random forest accuracy:", np.mean(cv_scores))

Random forest accuracy: 0.7587570621468926


In [15]:
scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train)
X_test_svm = scaler.transform(X_test)
svm_model = SVC(random_state=42, C=3, gamma=0.5)
cv_scores = cross_val_score(svm_model, X_train_svm, y_train, cv=5, scoring='accuracy')
print("SVM accuracy:", np.mean(cv_scores))

SVM accuracy: 0.7687005649717514


In [16]:
xgb_model = XGBClassifier(random_state=42, n_estimators = 380, learning_rate = 0.3, max_depth = 4, alpha = 0.2, eval_metric = 'logloss')
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy')
print("XGB accuracy:", np.mean(cv_scores))

XGB accuracy: 0.7487005649717514


In [17]:
dt_model.fit(X_train, y_train)
y_dt_pred = dt_model.predict(X_test)
dt_test_acc = accuracy_score(y_test, y_dt_pred)
precision = precision_score(y_test, y_dt_pred, average='weighted')
recall = recall_score(y_test, y_dt_pred, average='weighted')
f1 = f1_score(y_test, y_dt_pred, average='weighted')
print("Decision Tree Test Accuracy:", dt_test_acc)
print("Decision Tree Precision:", precision)
print("Decision Tree Recall:", recall)
print("Decision Tree F1 Score:", f1)

Decision Tree Test Accuracy: 0.7466666666666667
Decision Tree Precision: 0.7337622377622378
Decision Tree Recall: 0.7466666666666667
Decision Tree F1 Score: 0.7370583633234236


In [18]:
rf_model.fit(X_train, y_train)
y_rf_pred = rf_model.predict(X_test)
rf_test_acc = accuracy_score(y_test, y_rf_pred)
precision = precision_score(y_test, y_rf_pred, average='weighted')
recall = recall_score(y_test, y_rf_pred, average='weighted')
f1 = f1_score(y_test, y_rf_pred, average='weighted')
print("Random Forest Test Accuracy:", rf_test_acc)
print("Random Forest Precision:", precision)
print("Random Forest Recall:", recall)
print("Random Forest F1 Score:", f1)

Random Forest Test Accuracy: 0.76
Random Forest Precision: 0.7561212121212122
Random Forest Recall: 0.76
Random Forest F1 Score: 0.7541146403797007


In [19]:
svm_model.fit(X_train_svm, y_train)
y_svm_pred = svm_model.predict(X_test_svm)
svm_test_acc = accuracy_score(y_test, y_svm_pred)
precision = precision_score(y_test, y_svm_pred, average='weighted')
recall = recall_score(y_test, y_svm_pred, average='weighted')
f1 = f1_score(y_test, y_svm_pred, average='weighted')
print("SVM Test Accuracy:", svm_test_acc)
print("SVM Precision:", precision)
print("SVM Recall:", recall)
print("SVM F1 Score:", f1)

SVM Test Accuracy: 0.7733333333333333
SVM Precision: 0.7839478710257275
SVM Recall: 0.7733333333333333
SVM F1 Score: 0.7709501457278723


In [20]:
xgb_model.fit(X_train, y_train)
y_xgb_pred = xgb_model.predict(X_test)
xgb_test_acc = accuracy_score(y_test, y_xgb_pred)
precision = precision_score(y_test, y_xgb_pred, average='weighted')
recall = recall_score(y_test, y_xgb_pred, average='weighted')
f1 = f1_score(y_test, y_xgb_pred, average='weighted')
print("XGBoost Test Accuracy:", xgb_test_acc)
print("XGBoost Precision:", precision)
print("XGBoost Recall:", recall)
print("XGBoost F1 Score:", f1)

XGBoost Test Accuracy: 0.7466666666666667
XGBoost Precision: 0.7541824751580848
XGBoost Recall: 0.7466666666666667
XGBoost F1 Score: 0.7493684210526315


In [21]:
voting = VotingClassifier(
    estimators=[("DT", dt_model), ("RF", rf_model), ("SVM", svm_model), ("XGB", xgb_model)],
    voting="hard"
)

voting.fit(X_train, y_train)
y_vote_pred = voting.predict(X_test)
voting_acc = accuracy_score(y_test, y_vote_pred)
precision = precision_score(y_test, y_vote_pred, average='weighted')
recall = recall_score(y_test, y_vote_pred, average='weighted')
f1 = f1_score(y_test, y_vote_pred, average='weighted')

print("Voting Classifier Test Accuracy:", voting_acc)
print("Voting Classifier Precision:", precision)
print("Voting Classifier Recall:", recall)
print("Voting Classifier F1 Score:", f1)

Voting Classifier Test Accuracy: 0.7333333333333333
Voting Classifier Precision: 0.7294149659863944
Voting Classifier Recall: 0.7333333333333333
Voting Classifier F1 Score: 0.7173440285204992
